In [3]:
import json
import time
from pathlib import Path

import ee
import geopandas as gpd
import shapely

# === KONFIGURASI ===
PROJECT_ID = "project2-503507"
GADM_LOCAL_PATH = Path("D:\\qgis\\gadm41_IDN.gpkg")  # Bisa .shp, .gpkg, atau .geojson
GADM_FIELD = "NAME_2"                            # GADM umumnya 'NAME_2'; BIG/BPS umumnya 'WADMKK'
ASSET_ID = f"projects/{PROJECT_ID}/assets/aoi_kutai_kartanegara"
OVERWRITE_ASSET = False


def init_gee(project_id: str) -> None:
    try:
        ee.Initialize(project=project_id)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=project_id)


def load_kukar_geometry(
    path: Path,
    field_name: str = "NAME_2",
    simplify_tol_m: float = 30.0,
) -> dict:
    """Membaca file GADM lokal, memfilter Kutai Kartanegara, dan mengembalikan GeoJSON dict ringan."""
    if not path.exists():
        raise FileNotFoundError(f"File GADM tidak ditemukan: {path}")

    read_kwargs = {}
    # Deteksi otomatis layer kabupaten jika input berupa .gpkg multi-layer (GADM v4.1)
    if path.suffix.lower() == ".gpkg":
        layers = gpd.list_layers(path)["name"].tolist()
        lvl2 = [lyr for lyr in layers if "2" in lyr]
        if lvl2:
            read_kwargs["layer"] = lvl2[0]

    gdf = gpd.read_file(path, **read_kwargs)
    if field_name not in gdf.columns:
        raise KeyError(f"Kolom '{field_name}' tidak ada. Kolom tersedia: {list(gdf.columns)}")

    # Filter fleksibel (menangani 'Kutai Kartanegara' maupun 'KutaiKartanegara')
    target_mask = (
        gdf[field_name]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.lower()
        .eq("kutaikartanegara")
    )
    kukar = gdf[target_mask].copy()

    if kukar.empty:
        sample_vals = gdf[field_name].dropna().unique()[:15].tolist()
        raise ValueError(f"Kutai Kartanegara tidak ditemukan di '{field_name}'. Contoh nilai: {sample_vals}")

    if kukar.crs is None:
        kukar = kukar.set_crs("EPSG:4326")

    # Gabungkan menjadi 1 geometri utuh, pastikan 2D valid, dan simplifikasi ringan agar lolos limit API GEE
    kukar_eq = kukar.to_crs("EPSG:6933")
    kukar_eq["geometry"] = shapely.force_2d(kukar_eq.geometry.make_valid())
    kukar_eq = kukar_eq.dissolve()
    kukar_eq["geometry"] = kukar_eq.geometry.simplify(
        tolerance=simplify_tol_m, preserve_topology=True
    )

    kukar_wgs84 = kukar_eq.to_crs("EPSG:4326").reset_index(drop=True)
    kukar_clean = gpd.GeoDataFrame(
        {"NAME_2": ["Kutai Kartanegara"]},
        geometry=kukar_wgs84.geometry,
        crs="EPSG:4326",
    )

    geojson_raw = json.loads(kukar_clean.to_json())
    return {"type": "FeatureCollection", "features": geojson_raw["features"]}


def upload_aoi_to_gee_asset(
    geojson_dict: dict,
    asset_id: str,
    overwrite: bool = False,
    wait: bool = True,
) -> None:
    """Mengunggah GeoJSON FeatureCollection ke GEE Assets secara terprogram."""
    try:
        ee.data.getAsset(asset_id)
        exists = True
    except ee.EEException:
        exists = False

    if exists:
        if not overwrite:
            print(f"Asset sudah tersedia (skip upload): {asset_id}")
            return
        ee.data.deleteAsset(asset_id)
        print(f"Asset lama dihapus untuk ditimpa: {asset_id}")

    ee_fc = ee.FeatureCollection(geojson_dict)
    task = ee.batch.Export.table.toAsset(
        collection=ee_fc,
        description="upload_aoi_kukar",
        assetId=asset_id,
    )
    task.start()
    print(f"Mengunggah AOI ke GEE Asset: {asset_id} ...")

    if not wait:
        return

    while True:
        status = task.status()
        state = status["state"]
        if state == "COMPLETED":
            print(f"Upload selesai! Asset siap digunakan: {asset_id}")
            break
        if state in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Upload asset gagal ({state}): {status.get('error_message')}")
        time.sleep(5)


# --- EKSEKUSI CELL 1 ---
init_gee(PROJECT_ID)
kukar_geojson = load_kukar_geometry(GADM_LOCAL_PATH, field_name=GADM_FIELD, simplify_tol_m=30.0)
upload_aoi_to_gee_asset(kukar_geojson, asset_id=ASSET_ID, overwrite=OVERWRITE_ASSET, wait=True)

Mengunggah AOI ke GEE Asset: projects/project2-503507/assets/aoi_kutai_kartanegara ...
Upload selesai! Asset siap digunakan: projects/project2-503507/assets/aoi_kutai_kartanegara


In [5]:
# --- CELL 2: CEK VISUAL POLYGON BATAS KUTAI KARTANEGARA DI GEE ---
import geemap

aoi = ee.FeatureCollection(ASSET_ID)

jumlah_fitur = aoi.size().getInfo()
luas_km2 = aoi.geometry().area(maxError=1).divide(1e6).getInfo()
batas = aoi.geometry().bounds().getInfo()

print(f"Asset : {ASSET_ID}")
print(f"Jumlah fitur : {jumlah_fitur}")
print(f"Luas perkiraan : {luas_km2:,.2f} km²")
print("Bounds :", batas["coordinates"])

m = geemap.Map(center=[-0.45, 116.5], zoom=8)
m.add_basemap("SATELLITE")
m.add_basemap("OpenStreetMap")

m.addLayer(
    aoi,
    {
        "color": "FF0000",
        "width": 3,
        "fillColor": "FF000033",
    },
    "Batas Kutai Kartanegara",
)

m.centerObject(aoi, 8)
m.addLayerControl()
m

Asset : projects/project2-503507/assets/aoi_kutai_kartanegara
Jumlah fitur : 1
Luas perkiraan : 25,296.99 km²
Bounds : [[[115.46199039603486, -1.1403479363674447], [117.63493297992882, -1.1403479363674447], [117.63493297992882, 1.5335918954682253], [115.46199039603486, 1.5335918954682253], [115.46199039603486, -1.1403479363674447]]]


Map(center=[0.09064757347157645, 116.5095770194419], controls=(WidgetControl(options=['position', 'transparent…

In [6]:
import ee


def build_kukar_deforestation_fc(
    aoi: ee.Geometry,
    year_start: int = 2020,
    year_end: int = 2025,
    scale_m: int = 30,
    min_area_m2: float = 50_000.0,
) -> ee.FeatureCollection:
    """
    Mendeteksi transisi tutupan lahan hutan (Trees = 1) pada year_start
    menjadi lahan terbuka/pascatambang (Bare = 7) atau kebun/semak (Grass=2, Crops=4, Shrub=5)
    pada year_end, lalu mempoligonisasinya.
    """
    dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(aoi).select("label")

    # Komposit modus tahunan untuk mereduksi gangguan awan tropis
    dw_t1 = dw.filterDate(f"{year_start}-01-01", f"{year_start}-12-31").mode().clip(aoi)
    dw_t2 = dw.filterDate(f"{year_end}-01-01", f"{year_end}-12-31").mode().clip(aoi)

    is_forest_t1 = dw_t1.eq(1)

    # 1 = Lahan Terbuka / Indikasi Pascatambang (bare)
    # 2 = Kebun / Semak / Vegetasi Rendah (grass, crops, shrub_and_scrub)
    target_t2 = dw_t2.remap([7, 2, 4, 5], [1, 2, 2, 2], 0)
    deforestation = target_t2.updateMask(is_forest_t1).selfMask()

    # 1. Morphological smoothing (3x3 focal mode)
    smoothed = (
        deforestation.unmask(0)
        .focalMode(radius=1, kernelType="square", units="pixels")
        .selfMask()
        .toInt8()
    )

    # 2. Pre-filtering raster (80% dari min_area_m2) untuk menekan memori reduceToVectors
    pixel_area_m2 = scale_m * scale_m
    min_pixels = max(10, min(250, int((min_area_m2 * 0.8) / pixel_area_m2)))

    connected = smoothed.connectedPixelCount(maxSize=256, eightConnected=True)
    cleaned = smoothed.updateMask(connected.gte(min_pixels)).rename("kelas_id")

    # 3. Poligonisasi (Raster -> Vektor)
    vectors = cleaned.reduceToVectors(
        geometry=aoi,
        crs="EPSG:4326",
        scale=scale_m,
        geometryType="polygon",
        eightConnected=False,
        labelProperty="kelas_id",
        bestEffort=True,
        maxPixels=1e10,
    )

    return vectors


def export_geojson_to_drive(
    fc: ee.FeatureCollection,
    description: str = "deforestasi_kukar_2020_2025_min5ha",
    folder: str = "GEE_Reboisasi_Kukar",
) -> ee.batch.Task:
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=description,
        folder=folder,
        fileNamePrefix=description,
        fileFormat="GeoJSON",
    )
    task.start()
    print(f"Task ekspor dimulai: {description} | Status: {task.status()['state']}")
    print("Pantau progres ekspor di https://code.earthengine.google.com/tasks")
    return task


# --- EKSEKUSI CELL 2 ---
aoi_kukar = ee.FeatureCollection(ASSET_ID).geometry()

deforest_vectors = build_kukar_deforestation_fc(
    aoi=aoi_kukar,
    year_start=2020,
    year_end=2025,
    scale_m=30,
    min_area_m2=50_000.0,  # Minimal ~5 ha
)

export_task = export_geojson_to_drive(
    deforest_vectors,
    description="deforestasi_kukar_2020_2025_min5ha",
    folder="GEE_Reboisasi_Kukar",
)

Task ekspor dimulai: deforestasi_kukar_2020_2025_min5ha | Status: READY
Pantau progres ekspor di https://code.earthengine.google.com/tasks


### Cell 4: Enrichment Parameter Fisik Lingkungan (Elevasi, Slope, Curah Hujan, Jenis Tanah)

Berdasarkan validasi metadata 5 layer aktif pada proyek QGIS (`ZAKAT DEFORESTASI.qgz`):
1. **Keselarasan CRS:** Seluruh 5 layer input berada pada `EPSG:4326`. Ekstraksi raster dilakukan langsung pada CRS asli raster (tanpa reprojection raster yang memicu distorsi interpolasi), sedangkan perhitungan luas dan overlay jenis tanah diproyeksikan sementara ke Equal-Area `EPSG:6933` sebelum output dikembalikan ke `EPSG:4326`.
2. **FABDEM Elevasi & Slope (30 m, `float32`, tiled 256x256 LZW):** Header `nodata` bernilai `None` dan piksel luar AOI bernilai IEEE `NaN`. Pembacaan menggunakan *windowed masking* (`rasterio.mask(..., crop=True, filled=True, nodata=np.nan)`) agar hemat RAM dan menghindari kontaminasi nol palsu. `slope_std` memakai `ddof=0` agar 53 poligon berukuran 1 piksel bernilai `0.0` (bukan `NaN`), konsisten dengan QGIS Zonal Statistics.
3. **ERA5-Land Curah Hujan Tahunan (`buffered_0p1deg`, resolusi ~0,1°):** Karena piksel kasar (~11,1 km), mayoritas poligon kecil tidak memuat pusat piksel. Pipeline otomatis melakukan fallback sampling menggunakan `representative_point()` (*point-on-surface*, dijamin berada di dalam poligon) sehingga menghasilkan 0 nilai `NaN`.
4. **Jenis Tanah (`jenis_tanah_kukar_ikn`, kolom `ket_jenis`):** Perbaikan topologi dengan `make_valid()` dan overlay `gpd.overlay(..., how='intersection')` pada `EPSG:6933`, diikuti agregasi luas per `(_poly_idx, ket_jenis)` (penting karena kelas `'Sungai'` terdiri dari 3 fitur terpisah). Poligon tepi batas yang tidak beririsan langsung ditangani dengan fallback `sjoin_nearest`.


In [1]:
# --- CELL 4: ENRICHMENT PARAMETER FISIK LINGKUNGAN DEFORESTASI KUKAR ---
# Menggunakan modul pipeline tervalidasi: scripts/enrich_deforestation_params.py
import sys
from pathlib import Path

WORKSPACE_ROOT = Path(r"D:\qgis\ZAKAT DEFORESTASI")
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from scripts.enrich_deforestation_params import run_enrichment_pipeline

gdf_enriched, laporan_verifikasi_md = run_enrichment_pipeline(
    workspace_root=WORKSPACE_ROOT,
    output_dir=WORKSPACE_ROOT / "data" / "processed",
)

print(laporan_verifikasi_md)
gdf_enriched.drop(columns="geometry").head(5)


21:36:13 | INFO    | Memuat layer deforestasi: C:\Users\najwa\Downloads\deforestasi_kukar_2020_2025_min5ha.geojson
21:36:16 | INFO    | Deforestasi dimuat: 2102 poligon | CRS=EPSG:4326 | Kolom=['id', 'count', 'kelas_id', 'Luas_m2', 'geometry']
21:36:17 | INFO    | Mengekstrak elevasi FABDEM (windowed reading): C:\Users\najwa\Downloads\fabdem_elev_kukar_30m.tif
21:36:26 | INFO    | Selesai elevasi FABDEM: {'zonal_center': 2102, 'fallback_all_touched': 0, 'fallback_rep_point': 0}
21:36:26 | INFO    | Mengekstrak kelerengan FABDEM (windowed reading): C:\Users\najwa\Downloads\fabdem_slope_kukar_30m.tif
21:36:36 | INFO    | Selesai slope FABDEM: {'zonal_center': 2102, 'fallback_all_touched': 0, 'fallback_rep_point': 0, 'single_pixel_polygons': 53}
21:36:36 | INFO    | Mengekstrak curah hujan ERA5-Land (0.1 deg + fallback): D:\qgis\ZAKAT DEFORESTASI\era5_kukar_rain\ERA5Land_RainMean_mmYr_Kukar_buffered_0p1deg.tif
21:36:44 | INFO    | Selesai curah hujan ERA5-Land: {'zonal_center': 3, 'fallba

# Ringkasan Verifikasi Parameter Fisik — Deforestasi Kutai Kartanegara

## 1. Sumber Data & Validasi Integritas
- **Poligon Deforestasi:** `C:\Users\najwa\Downloads\deforestasi_kukar_2020_2025_min5ha.geojson`
- **Elevasi FABDEM (30 m):** `C:\Users\najwa\Downloads\fabdem_elev_kukar_30m.tif`
- **Slope FABDEM (30 m):** `C:\Users\najwa\Downloads\fabdem_slope_kukar_30m.tif`
- **Curah Hujan ERA5-Land (unsmoothed 0.1°):** `D:\qgis\ZAKAT DEFORESTASI\era5_kukar_rain\ERA5Land_RainMean_mmYr_Kukar_buffered_0p1deg.tif`
- **Jenis Tanah (kolom `ket_jenis`):** `D:\qgis\ZAKAT DEFORESTASI\data\jenis_tanah_kukar_ikn.geojson`
- **CRS Output:** `EPSG:4326` (overlay luas & jenis tanah diproses pada `EPSG:6933`)
- **Jumlah Poligon Awal vs Akhir:** `2,102` → `2,102` (rasio 1:1 terverifikasi)
- **Total Luas Poligon Deforestasi:** `43,765.90 ha`
- **Poligon Topografi Heterogen (`slope_std >= 8.0°`):** `12` (0.6%)
- **Waktu Eksekusi Pipeline:** `44.79 detik`

### Log Ekstraksi & Fallback
- **FABDEM Elevasi:** zo

,id,count,kelas_id,Luas_m2,luas_ha,elev_med,elev_rng,slope_med,slope_std,slope_p90,topo_heterogen,rain_mean,tanah_dom,tanah_dom_pct,tanah_n_kelas
0,+428854+3720,53,1,47373.63,4.74,1039.9,132.4,25.8,4.9,30.7,False,3044.7,"Komplek Padsolik Merah Kuning,Latosol & Litosol",100.0,1
1,+428893+4692,1,2,893.76,0.09,1089.8,0.0,6.4,0.0,6.4,False,2972.0,"Komplek Padsolik Merah Kuning,Latosol & Litosol",100.0,1
2,+428921+4744,400,2,357504.16,35.75,1157.9,171.2,13.7,5.9,22.0,False,2972.0,"Komplek Padsolik Merah Kuning,Latosol & Litosol",100.0,1
3,+429116+4671,46,1,41113.21,4.11,933.9,0.4,0.1,0.2,0.3,False,2972.0,"Komplek Padsolik Merah Kuning,Latosol & Litosol",100.0,1
4,+429258+3952,82,1,73293.66,7.33,783.1,67.5,11.7,5.9,20.6,False,2954.8,"Komplek Padsolik Merah Kuning,Latosol & Litosol",100.0,1


In [3]:
import ee
import geemap

# === KONFIGURASI ===
PROJECT_ID = "project2-503507"
ASSET_ID = f"projects/{PROJECT_ID}/assets/aoi_kutai_kartanegara"
YEAR_START = 2020
YEAR_END = 2025
CS_THRESHOLD = 0.60  # Ambang batas kebersihan awan Cloud Score+ (0.55 - 0.65)

# Pastikan GEE sudah terinisialisasi pada sesi kernel saat ini
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# Menggunakan aoi_kukar / ASSET_ID yang sudah ada di kernel
if "aoi_kukar" not in globals():
    aoi_kukar = ee.FeatureCollection(ASSET_ID).geometry()


def get_s2_rgb(aoi: ee.Geometry, year: int, cs_thresh: float = 0.60) -> ee.Image:
    """Membuat komposit median tahunan Sentinel-2 RGB bebas awan menggunakan Cloud Score+."""
    start, end = f"{year}-01-01", f"{year}-12-31"

    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
    )
    cs_plus = (
        ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
    )

    def mask_and_scale(img: ee.Image) -> ee.Image:
        clear = img.select("cs_cdf").gte(cs_thresh)
        return img.select(["B4", "B3", "B2"]).multiply(0.0001).updateMask(clear)

    return s2.linkCollection(cs_plus, ["cs_cdf"]).map(mask_and_scale).median().clip(aoi)


def get_dw_deforestation_layers(
    aoi: ee.Geometry, year_start: int, year_end: int
) -> tuple[ee.Image, ee.Image]:
    """
    Mengembalikan dua raster deforestasi Dynamic World (raw & smoothed 3x3):
      1 = Lahan Terbuka / Pascatambang (bare)
      2 = Kebun / Semak / Vegetasi Rendah (grass, crops, shrub)
    """
    dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(aoi).select("label")
    dw_t1 = dw.filterDate(f"{year_start}-01-01", f"{year_start}-12-31").mode().clip(aoi)
    dw_t2 = dw.filterDate(f"{year_end}-01-01", f"{year_end}-12-31").mode().clip(aoi)

    is_forest_t1 = dw_t1.eq(1)
    target_t2 = dw_t2.remap([7, 2, 4, 5], [1, 2, 2, 2], 0)

    dw_raw = target_t2.updateMask(is_forest_t1).selfMask().toInt8()
    dw_smoothed = (
        dw_raw.unmask(0)
        .focalMode(radius=1, kernelType="square", units="pixels")
        .selfMask()
        .toInt8()
    )
    return dw_raw, dw_smoothed


# 1. Siapkan layer
s2_t1 = get_s2_rgb(aoi_kukar, YEAR_START, CS_THRESHOLD)
s2_t2 = get_s2_rgb(aoi_kukar, YEAR_END, CS_THRESHOLD)
dw_def_raw, dw_def_smooth = get_dw_deforestation_layers(aoi_kukar, YEAR_START, YEAR_END)

aoi_outline = ee.Image().byte().paint(
    featureCollection=ee.FeatureCollection([ee.Feature(aoi_kukar)]),
    color=1,
    width=2,
)

# 2. Parameter visualisasi
vis_s2 = {"bands": ["B4", "B3", "B2"], "min": 0.02, "max": 0.25, "gamma": 1.2}
vis_dw_def = {"min": 1, "max": 2, "palette": ["#EF4444", "#F59E0B"]}  # 1: Merah (Bare), 2: Oranye (Semak/Kebun)

# 3. Render peta interaktif
m = geemap.Map()
m.centerObject(aoi_kukar, 10)

m.addLayer(s2_t1, vis_s2, f"Sentinel-2 RGB {YEAR_START} (Awal)", True)
m.addLayer(s2_t2, vis_s2, f"Sentinel-2 RGB {YEAR_END} (Akhir)", True)
m.addLayer(dw_def_raw, vis_dw_def, f"DW Deforestasi Raw ({YEAR_START}-{YEAR_END})", False)
m.addLayer(dw_def_smooth, vis_dw_def, f"DW Deforestasi Smoothed ({YEAR_START}-{YEAR_END})", True)
m.addLayer(aoi_outline, {"palette": ["#00FFFF"]}, "Batas Kutai Kartanegara", True)

m.add_legend(
    title="Deforestasi Dynamic World",
    legend_dict={
        "Lahan Terbuka / Pascatambang (Kelas 1)": "#EF4444",
        "Kebun / Semak / Vegetasi Rendah (Kelas 2)": "#F59E0B",
    },
)

m

Map(center=[0.0906475734715811, 116.50957701944193], controls=(WidgetControl(options=['position', 'transparent…